# PHANOTATE-rs ORF score model playground

This notebook trains and benchmarks candidate ORF scoring models for
`phanotate-rs --model`. The goal is to replace (or improve on) the default
PHANOTATE heuristic edge weight with a learned score.

## Cross-validation workflow

Because ORFs from the same genome are highly correlated, we split by
**genome** (not by individual ORF):

1. Load annotated GenBank files.
2. Extract per-ORF features with `phanotate-rs --export-features`.
3. Label each ORF by overlap with the GenBank `CDS` coordinates.
4. For each CV fold:
   - Train a model on the training genomes.
   - Export a JSON model compatible with `--model`.
   - Run `phanotate-rs --model <json>` on each validation genome.
   - Compare predicted genes to GenBank annotations.
5. Pick the best model and re-train on all data.

Only linear models with explicit coefficients can be exported to the JSON
format consumed by Rust, but you can benchmark any classifier inside the
notebook.

In [ ]:
import json
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import seaborn as sns

# Allow importing the training-script helpers
sys.path.insert(0, ".")
from scripts.train_orf_score_model import (
    parse_genbank,
    _majority_table,
    _find_phanotate_binary,
    SUPPORTED_TABLES,
    FEATURE_NAMES,
)

BINARY = _find_phanotate_binary()
print("Using binary:", BINARY)

# Path to annotated GenBank files
GENBANK_DIR = Path("tests/golden/annotgenomes")
if not GENBANK_DIR.exists():
    raise FileNotFoundError(f"Directory not found: {GENBANK_DIR}")

gb_paths = sorted(GENBANK_DIR.rglob("*.gb"))
print(f"Found {len(gb_paths)} GenBank files")

## 1. Extract features and labels for every genome

We call `phanotate-rs --export-features` for each GenBank file. The first
two columns of the TSV are `start`/`stop`; the remaining columns are the
`OrfFeatures` vector. Labels are derived from GenBank `CDS` features.

In [ ]:
def load_genome_features(path: Path, table_override: int | None = None) -> pd.DataFrame:
    '''Return a DataFrame of ORF features + labels for one GenBank file.'''
    _, entries = parse_genbank(str(path))

    if table_override is not None:
        table = table_override
    else:
        table = _majority_table(entries)
        if table not in SUPPORTED_TABLES:
            raise ValueError(f"Unsupported table {table} in {path}")

    # Build label set from CDS coordinates
    labels = set()
    for s, e, strand, _ in entries:
        if strand == "+":
            if e - s + 1 >= 3:
                labels.add((s, e - 2))
        else:
            labels.add((s, e))

    with tempfile.NamedTemporaryFile(mode="w", suffix=".tsv", delete=False) as tmp:
        features_path = tmp.name

    try:
        subprocess.run(
            [BINARY, "-i", str(path), "-g", str(table), "--export-features", features_path],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        df = pd.read_csv(features_path, sep="\t")
    finally:
        Path(features_path).unlink(missing_ok=True)

    df["genome"] = path.stem
    df["table"] = table
    df["is_gene"] = df.apply(lambda r: 1 if (int(r["start"]), int(r["stop"])) in labels else 0, axis=1)
    return df


records = []
skipped = []
for p in gb_paths:
    try:
        records.append(load_genome_features(p))
    except Exception as exc:
        skipped.append((p.name, str(exc)))

if skipped:
    print("Skipped files:")
    for name, reason in skipped:
        print(f"  {name}: {reason}")

df = pd.concat(records, ignore_index=True)
print(f"Total ORFs: {len(df)}; genes: {df['is_gene'].sum()}; genomes: {df['genome'].nunique()}")
df.head()

## 2. Quick EDA

Look at feature distributions and how they differ between genes and
non-genes.

In [ ]:
Path("outputs").mkdir(exist_ok=True)

fig, axes = plt.subplots(4, 4, figsize=(14, 12))
axes = axes.ravel()
for i, col in enumerate(FEATURE_NAMES):
    ax = axes[i]
    df[df["is_gene"] == 1][col].hist(ax=ax, bins=30, alpha=0.6, label="gene")
    df[df["is_gene"] == 0][col].hist(ax=ax, bins=30, alpha=0.6, label="non-gene")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.savefig("outputs/feature_distributions.png", dpi=150)
plt.show()

In [ ]:
corr = df[FEATURE_NAMES + ["is_gene"]].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.savefig("outputs/feature_correlations.png", dpi=150)
plt.show()

## 3. Genome-stratified cross-validation

We split by `genome`. For each fold we:

- standardise features on the training ORFs,
- fit one or more candidate models,
- export the best linear model to JSON,
- run `phanotate-rs --model` on each validation genome,
- compare predicted gene coordinates to the GenBank CDS set.

The helper below converts a fitted linear model into the JSON consumed by
`src/orf_score_model.rs`.

In [ ]:
def export_linear_model(model, mean: np.ndarray, std: np.ndarray, path: Path) -> None:
    '''Export a fitted scikit-learn linear classifier to the Rust JSON format.'''
    # model.coef_ shape is (1, n_features) for binary LogisticRegression
    coef = np.asarray(model.coef_).ravel()
    out = {
        "version": 1,
        "num_features": len(coef),
        "coeffs": coef.tolist(),
        "mean": mean.tolist(),
        "std": std.tolist(),
    }
    path.write_text(json.dumps(out, indent=2))


def predict_genes(genbank_path: Path, model_path: Path, table: int) -> set:
    '''Run phanotate-rs --model and return predicted (start, stop) gene coordinates.'''
    result = subprocess.run(
        [BINARY, "-i", str(genbank_path), "-g", str(table), "--model", str(model_path), "-f", "sco"],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    genes = set()
    for line in result.stdout.splitlines():
        if line.startswith("#") or not line.strip():
            continue
        parts = line.split("\t")
        if len(parts) < 3:
            continue
        genes.add((int(parts[0]), int(parts[1])))
    return genes


def genome_cds_set(genbank_path: Path) -> set:
    '''Return the GenBank CDS coordinate set for comparison.'''
    _, entries = parse_genbank(str(genbank_path))
    cds = set()
    for s, e, strand, _ in entries:
        if strand == "+" and e - s + 1 >= 3:
            cds.add((s, e - 2))
        else:
            cds.add((s, e))
    return cds


def gene_metrics(pred: set, true: set) -> dict:
    tp = len(pred & true)
    fp = len(pred - true)
    fn = len(true - pred)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

In [ ]:
genome_ids = df["genome"].unique()
n_folds = min(3, len(genome_ids))
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

fold_results = []
fold_idx = 1
for train_idx, val_idx in kf.split(genome_ids):
    train_genomes = genome_ids[train_idx]
    val_genomes = genome_ids[val_idx]

    train_df = df[df["genome"].isin(train_genomes)]
    val_df = df[df["genome"].isin(val_genomes)]

    X_train = train_df[FEATURE_NAMES].values
    y_train = train_df["is_gene"].values
    X_val = val_df[FEATURE_NAMES].values
    y_val = val_df["is_gene"].values

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0.0] = 1.0

    # Candidate 1: logistic regression (exportable)
    lr = LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs")
    lr.fit((X_train - mean) / std, y_train)
    val_prob_lr = lr.predict_proba((X_val - mean) / std)[:, 1]

    # Candidate 2: random forest (not exportable, but useful benchmark)
    rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train)
    val_prob_rf = rf.predict_proba(X_val)[:, 1]

    # ORF-level classification metrics
    orf_metrics = {
        "fold": fold_idx,
        "n_train_genomes": len(train_genomes),
        "n_val_genomes": len(val_genomes),
        "lr_auc": roc_auc_score(y_val, val_prob_lr),
        "rf_auc": roc_auc_score(y_val, val_prob_rf),
    }

    # Export the logistic-regression model and run the full PHANOTATE pipeline
    model_path = Path(f"outputs/model_fold{fold_idx}.json")
    export_linear_model(lr, mean, std, model_path)

    pipeline_metrics = []
    for p in gb_paths:
        if p.stem not in val_genomes:
            continue
        table = int(df[df["genome"] == p.stem]["table"].iloc[0])
        pred = predict_genes(p, model_path, table)
        true = genome_cds_set(p)
        m = gene_metrics(pred, true)
        m["genome"] = p.stem
        pipeline_metrics.append(m)

    if pipeline_metrics:
        pm = pd.DataFrame(pipeline_metrics)
        orf_metrics["pipeline_precision"] = pm["precision"].mean()
        orf_metrics["pipeline_recall"] = pm["recall"].mean()
        orf_metrics["pipeline_f1"] = pm["f1"].mean()

    fold_results.append(orf_metrics)
    print(f"Fold {fold_idx}: {orf_metrics}")
    fold_idx += 1

pd.DataFrame(fold_results)

## 4. Compare candidate models

Plot ORF-level ROC and PR curves on held-out genomes. The exportable model
is logistic regression; Random Forest and XGBoost are shown for comparison.

In [ ]:
# Simple held-out split for model comparison plots
np.random.seed(42)
val_genomes_plot = np.random.choice(genome_ids, size=max(1, len(genome_ids) // 5), replace=False)
train_genomes_plot = [g for g in genome_ids if g not in val_genomes_plot]

train_df_p = df[df["genome"].isin(train_genomes_plot)]
val_df_p = df[df["genome"].isin(val_genomes_plot)]
X_train_p = train_df_p[FEATURE_NAMES].values
y_train_p = train_df_p["is_gene"].values
X_val_p = val_df_p[FEATURE_NAMES].values
y_val_p = val_df_p["is_gene"].values

mean_p = X_train_p.mean(axis=0)
std_p = X_train_p.std(axis=0)
std_p[std_p == 0.0] = 1.0

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs"),
    "RandomForest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for name, m in models.items():
    if name == "LogisticRegression":
        m.fit((X_train_p - mean_p) / std_p, y_train_p)
        prob = m.predict_proba((X_val_p - mean_p) / std_p)[:, 1]
    else:
        m.fit(X_train_p, y_train_p)
        prob = m.predict_proba(X_val_p)[:, 1]
    fpr, tpr, _ = roc_curve(y_val_p, prob)
    prec, rec, _ = precision_recall_curve(y_val_p, prob)
    ax1.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_val_p, prob):.3f})")
    ax2.plot(rec, prec, label=f"{name}")

ax1.plot([0, 1], [0, 1], "k--")
ax1.set_xlabel("False positive rate")
ax1.set_ylabel("True positive rate")
ax1.set_title("ROC curve")
ax1.legend()

ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall curve")
ax2.legend()

plt.tight_layout()
plt.savefig("outputs/model_comparison.png", dpi=150)
plt.show()

## 5. Train final model and export

Retrain the chosen model on all genomes and write the JSON file that can be
passed to `phanotate-rs --model`.

In [ ]:
X_all = df[FEATURE_NAMES].values
y_all = df["is_gene"].values

mean_all = X_all.mean(axis=0)
std_all = X_all.std(axis=0)
std_all[std_all == 0.0] = 1.0

final_model = LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs")
final_model.fit((X_all - mean_all) / std_all, y_all)

final_path = Path("outputs/model_final.json")
export_linear_model(final_model, mean_all, std_all, final_path)
print(f"Exported final model to {final_path}")

# Show feature importance (coefficients)
coef_df = pd.DataFrame({"feature": FEATURE_NAMES, "coeff": final_model.coef_.ravel()})
coef_df = coef_df.sort_values("coeff", key=abs, ascending=False)
print(coef_df)

## 6. Validate the final model on a held-out genome

Pick a genome not seen during final training (or the same CV validation set)
and compare `--model` predictions to the GenBank annotations.

In [ ]:
# Example: validate on the first validation-genome from the last CV fold
example_genome = gb_paths[0]
example_table = int(df[df["genome"] == example_genome.stem]["table"].iloc[0])

default_pred = predict_genes(example_genome, Path("/dev/null"), example_table)
model_pred = predict_genes(example_genome, final_path, example_table)
true_set = genome_cds_set(example_genome)

print("Default PHANOTATE:", gene_metrics(default_pred, true_set))
print("Learned model:", gene_metrics(model_pred, true_set))

## Next steps

- Try different feature sets or add engineered features in `src/ml_features.rs`.
- Experiment with class weights, regularisation, or alternative linear models.
- Use the exported `outputs/model_final.json` with PHANOTATE-rs:
  ```bash
  ./target/release/phanotate-rs -i genome.fasta --model outputs/model_final.json -f sco
  ```